# Native CLM Architecture Lab

This is the single long-lived notebook for Native CLM architecture search.

Primary objective:

\[
\min_A \; \mathrm{NLL}_{\mathrm{NTP}}(A)
\]

under controlled parameter, data, token, and compute budgets.

The first milestone compares a modern ~2M decoder-only Transformer (`T1`) against a parameter-matched ~2M native CLM scaffold (`C0`), then adds one CLM architectural change at a time.

**Important:** continual-learning mechanisms (growth, mitosis, rollback, online writes) are intentionally excluded from the first phase.


## Protocol

For a direct comparison keep constant:

- tokenizer and vocabulary
- dataset revision/splits
- context length
- train-token budget
- optimizer and LR schedule
- seed set

Always report parameters, validation NLL/PPL, estimated training FLOPs, estimated inference FLOPs/token, throughput, latency, and peak memory.

Development runs are for iteration. Freeze a promoted candidate before untouched confirmation seeds. Export promoted records to `results/NCLM-XXX/`.


In [ ]:
from dataclasses import dataclass, asdict
from pathlib import Path
import json, math, platform, random

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE, "torch:", torch.__version__)


In [ ]:
@dataclass(frozen=True)
class LabConfig:
    seed: int = 91001
    vocab_size: int = 2048
    context_length: int = 128

    # Modern Transformer baseline (~2M).
    transformer_d_model: int = 192
    transformer_layers: int = 4
    n_heads: int = 6
    n_kv_heads: int = 2
    ffn_mult: float = 2.5

    # Native CLM engineering scaffold (~2M).
    # Width is chosen only for parameter matching, not as an architectural claim.
    clm_d_model: int = 288
    clm_steps: tuple = (4, 4, 4)
    clm_windows: tuple = (8, 32, 128)

CFG = LabConfig()
print(asdict(CFG))


In [ ]:
def seed_everything(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything(CFG.seed)

def count_parameters(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable

def nll_to_ppl(nll):
    return float(math.exp(float(nll)))


## T1 — modern small Transformer baseline

The baseline is deliberately modern rather than a historical GPT-Neo-style model:

- pre-norm RMSNorm
- RoPE
- SwiGLU
- grouped-query attention
- causal decoder
- tied embedding / LM-head weights

This is a small architecture-search baseline, not a claim that one fixed 2M configuration is globally optimal.


In [ ]:
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))
        self.eps = eps

    def forward(self, x):
        scale = x.pow(2).mean(-1, keepdim=True).add(self.eps).rsqrt()
        return x * scale * self.weight

def apply_rope(x):
    # x: [B,H,T,D], D must be even.
    B, H, T, D = x.shape
    assert D % 2 == 0
    half = D // 2
    inv = 1.0 / (10000 ** (torch.arange(0, half, device=x.device, dtype=x.dtype) / half))
    pos = torch.arange(T, device=x.device, dtype=x.dtype)
    ang = torch.einsum("t,d->td", pos, inv)
    sin, cos = ang.sin()[None, None, :, :], ang.cos()[None, None, :, :]
    x1, x2 = x[..., :half], x[..., half:]
    return torch.cat([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1)

class GQAAttention(nn.Module):
    def __init__(self, d_model, n_heads, n_kv_heads):
        super().__init__()
        assert d_model % n_heads == 0
        assert n_heads % n_kv_heads == 0
        self.n_heads = n_heads
        self.n_kv_heads = n_kv_heads
        self.head_dim = d_model // n_heads
        self.q = nn.Linear(d_model, n_heads * self.head_dim, bias=False)
        self.k = nn.Linear(d_model, n_kv_heads * self.head_dim, bias=False)
        self.v = nn.Linear(d_model, n_kv_heads * self.head_dim, bias=False)
        self.o = nn.Linear(n_heads * self.head_dim, d_model, bias=False)

    def forward(self, x):
        B, T, _ = x.shape
        q = self.q(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.k(x).view(B, T, self.n_kv_heads, self.head_dim).transpose(1, 2)
        v = self.v(x).view(B, T, self.n_kv_heads, self.head_dim).transpose(1, 2)
        q, k = apply_rope(q), apply_rope(k)
        rep = self.n_heads // self.n_kv_heads
        k = k.repeat_interleave(rep, dim=1)
        v = v.repeat_interleave(rep, dim=1)
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        return self.o(y.transpose(1, 2).contiguous().view(B, T, -1))

class SwiGLU(nn.Module):
    def __init__(self, d_model, hidden):
        super().__init__()
        self.gate = nn.Linear(d_model, hidden, bias=False)
        self.up = nn.Linear(d_model, hidden, bias=False)
        self.down = nn.Linear(hidden, d_model, bias=False)

    def forward(self, x):
        return self.down(F.silu(self.gate(x)) * self.up(x))

class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        d = cfg.transformer_d_model
        hidden = int(d * cfg.ffn_mult)
        self.n1 = RMSNorm(d)
        self.attn = GQAAttention(d, cfg.n_heads, cfg.n_kv_heads)
        self.n2 = RMSNorm(d)
        self.ff = SwiGLU(d, hidden)

    def forward(self, x):
        x = x + self.attn(self.n1(x))
        x = x + self.ff(self.n2(x))
        return x

class ModernTransformerLM(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        d = cfg.transformer_d_model
        self.tok = nn.Embedding(cfg.vocab_size, d)
        self.blocks = nn.ModuleList([TransformerBlock(cfg) for _ in range(cfg.transformer_layers)])
        self.norm = RMSNorm(d)
        self.lm_head = nn.Linear(d, cfg.vocab_size, bias=False)
        self.lm_head.weight = self.tok.weight

    def forward(self, ids):
        x = self.tok(ids)
        for block in self.blocks:
            x = block(x)
        return self.lm_head(self.norm(x))

T1 = ModernTransformerLM(CFG).to(DEVICE)
print("T1 params:", count_parameters(T1))


## C0 — Native CLM scaffold

`C0` is an **engineering scaffold**, not a claim that this is the historical or best CLM.

It intentionally starts with:

- shared Cell parameters across recurrent steps
- recurrent-step embeddings
- causal local receptive fields
- gated state update
- staged windows `(8, 32, 128)`

Its width is selected only to put parameter count close to `T1`. Before `NCLM-001` can be promoted, the exact historical Native CLM baseline/provenance must be reconstructed from repository artifacts and the baseline protocol frozen.


In [ ]:
def causal_window_mean(x, window):
    # x [B,T,C]; causal moving average including current position.
    B, T, C = x.shape
    cs = torch.cat(
        [torch.zeros(B, 1, C, device=x.device, dtype=x.dtype), x.cumsum(dim=1)],
        dim=1,
    )
    idx = torch.arange(T, device=x.device)
    start = (idx - window + 1).clamp_min(0)
    sums = cs[:, idx + 1] - cs[:, start]
    denom = (idx - start + 1).to(x.dtype)[None, :, None]
    return sums / denom

class SharedCell(nn.Module):
    def __init__(self, d_model, max_steps):
        super().__init__()
        self.norm = RMSNorm(d_model)
        self.step = nn.Embedding(max_steps, d_model)
        self.cand = SwiGLU(d_model * 2, d_model * 2)
        self.proj = nn.Linear(d_model * 2, d_model, bias=False)
        self.gate = nn.Linear(d_model * 2, d_model, bias=True)
        nn.init.constant_(self.gate.bias, 2.0)  # explicit hypothesis; must be ablated.

    def forward(self, h, msg, step_idx):
        step = self.step.weight[step_idx][None, None, :].expand_as(h)
        z = torch.cat([self.norm(h) + step, msg], dim=-1)
        candidate = torch.tanh(self.proj(self.cand(z)))
        keep = torch.sigmoid(self.gate(z))
        return keep * h + (1.0 - keep) * candidate

class NativeCLMLM(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        d = cfg.clm_d_model
        self.tok = nn.Embedding(cfg.vocab_size, d)
        self.max_steps = sum(cfg.clm_steps)
        self.cell = SharedCell(d, self.max_steps)
        self.norm = RMSNorm(d)
        self.lm_head = nn.Linear(d, cfg.vocab_size, bias=False)
        self.lm_head.weight = self.tok.weight

    def forward(self, ids):
        h = self.tok(ids)
        step_idx = 0
        for window, n_steps in zip(self.cfg.clm_windows, self.cfg.clm_steps):
            for _ in range(n_steps):
                msg = causal_window_mean(h, window)
                h = self.cell(h, msg, step_idx)
                step_idx += 1
        return self.lm_head(self.norm(h))

C0 = NativeCLMLM(CFG).to(DEVICE)
t1_params = count_parameters(T1)[0]
c0_params = count_parameters(C0)[0]
param_ratio = c0_params / t1_params
print("C0 params:", count_parameters(C0))
print("param ratio C0/T1:", param_ratio)
assert abs(param_ratio - 1.0) < 0.02, "initial baselines must be within 2% params"


## Smoke test

This verifies tensor plumbing and causal next-token loss only. It is **not** a scientific benchmark.

Replace the synthetic batch with the frozen dataset/tokenizer adapter before any promoted result.


In [ ]:
B = 2
x = torch.randint(0, CFG.vocab_size, (B, CFG.context_length), device=DEVICE)
with torch.no_grad():
    t_logits = T1(x[:, :-1])
    c_logits = C0(x[:, :-1])
    y = x[:, 1:]
    t_loss = F.cross_entropy(t_logits.reshape(-1, CFG.vocab_size), y.reshape(-1))
    c_loss = F.cross_entropy(c_logits.reshape(-1, CFG.vocab_size), y.reshape(-1))

print("smoke", {"T1_loss": float(t_loss), "C0_loss": float(c_loss)})


## Experiment registry

Add candidates here instead of creating new notebooks. A candidate is only marked `validated` after controlled multi-seed evidence.

Initial search order:

1. `C1`: gated-update ablation/optimization
2. `C2`: phase/step-conditioned recurrence
3. `C3`: small phase-specific modulation while keeping a shared Cell body
4. `C4`: receptive-field schedule search
5. later: dual state, adaptive recurrence, compute/communication separation, functional specialization


In [ ]:
EXPERIMENTS = [
    {"id": "T1", "family": "transformer", "status": "baseline", "change": "modern small decoder"},
    {"id": "C0", "family": "native-clm", "status": "baseline-scaffold", "change": "shared recurrent Cell scaffold"},
    {"id": "C1", "family": "native-clm", "status": "planned", "change": "gated-update study"},
    {"id": "C2", "family": "native-clm", "status": "planned", "change": "phase-conditioned recurrence"},
    {"id": "C3", "family": "native-clm", "status": "planned", "change": "phase-specific modulation"},
    {"id": "C4", "family": "native-clm", "status": "planned", "change": "receptive-field schedule"},
]
EXPERIMENTS


## Result row and immutable promotion helper

Promote only controlled runs. Never overwrite a previously promoted `NCLM-XXX` directory.


In [ ]:
def result_row(model_id, model, validation_nll, train_tokens, seed, **extra):
    total, trainable = count_parameters(model)
    row = {
        "model_id": model_id,
        "validation_nll": float(validation_nll),
        "validation_ppl": nll_to_ppl(validation_nll),
        "parameter_count_total": total,
        "parameter_count_trainable": trainable,
        "train_tokens": int(train_tokens),
        "context_length": CFG.context_length,
        "seed": int(seed),
    }
    row.update(extra)
    return row

def promote_run(run_id, config, metrics, root=Path("research/native-clm/results")):
    out = root / run_id
    if out.exists():
        raise FileExistsError(f"immutable promoted run already exists: {out}")
    out.mkdir(parents=True)
    (out / "config.json").write_text(json.dumps(config, indent=2, sort_keys=True))
    (out / "metrics.json").write_text(json.dumps(metrics, indent=2, sort_keys=True))
    env = {
        "python": platform.python_version(),
        "platform": platform.platform(),
        "torch": torch.__version__,
        "cuda": torch.version.cuda,
        "device": DEVICE,
    }
    (out / "environment.json").write_text(json.dumps(env, indent=2, sort_keys=True))
    (out / "README.md").write_text(
        f"# {run_id}\n\nPromoted from `native_clm_lab.ipynb`. "
        "Do not rewrite config/metrics after promotion.\n"
    )
    return out


## Next execution milestone

Before `NCLM-001` is promoted:

1. wire a frozen TinyStories (or chosen corpus) revision and one shared tokenizer into this notebook;
2. reconstruct and document the historical Native CLM baseline closely enough to define canonical `C0`;
3. freeze train-token budget, optimizer/schedule, precision, and evaluation policy;
4. implement and validate train/inference FLOP accounting for both families;
5. run development seeds;
6. freeze baseline protocol and untouched confirmation seeds;
7. only then promote the baseline and begin `C1`–`C4`.

The notebook remains the single architecture workbench throughout future revisions.
